# PV SCADA + Meteorological Dataset Builder — validated

Dataset sumber: Rostamipour, Alireza; Ghayeni, Amir Hossein (2026), “PV SCADA data”, Mendeley Data, V2, DOI: `10.17632/9fdvyjrf67.2`.

Tujuan notebook ini adalah membangun dataset terintegrasi pada resolusi SCADA 30 menit dengan mempertahankan variabel DC dan AC dari SCADA, serta mengagregasikan tiga weather station sebelum time alignment.

Perubahan utama:
- `dc_power_kw`, `ac_power_kw`, `dc_voltage_v`, `dc_current_a`, `module_temp_c`, `inverter_efficiency`, dan `status_code` dipertahankan.
- Nilai `dc_current_a` divalidasi terhadap hubungan fisik `P_DC = V_DC × I_DC / 1000`. Pada file sumber yang digunakan, angka current konsisten dengan skala kA meskipun nama kolom menyatakan A, sehingga nilai dikonversi ke A dengan faktor 1000.
- Tiga weather station diagregasi pada timestamp 15 menit menggunakan median, kemudian di-resample ke 30 menit.
- Alignment meteorologi ke SCADA dilakukan pada timestamp yang sama; tidak lagi memilih station secara arbitrer melalui `merge_asof`.
- Validasi unit dan physical consistency dilaporkan sebagai statistik/diagnostic. Tidak ada label fault/anomaly yang dibuat oleh notebook ini; anomaly detection tetap menjadi tanggung jawab model.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("PV SCADA data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = DATA_DIR / "Inverter SCADA.csv"
METEO_FILE = DATA_DIR / "Meteorological data.csv"
OUTPUT_FILE = OUTPUT_DIR / "pv_monitoring_dataset_validated.csv"
        


In [ ]:
scada = pd.read_csv(SCADA_FILE)
meteo = pd.read_csv(METEO_FILE)

print("SCADA shape:", scada.shape)
print("Meteorological shape:", meteo.shape)
print("\nSCADA columns:", scada.columns.tolist())
print("Meteorological columns:", meteo.columns.tolist())


## 1. Schema and timestamp validation

Validasi ini hanya memastikan struktur data sesuai sumber. Baris dengan timestamp invalid dibuang karena tidak dapat digunakan untuk time-series alignment.

In [ ]:
SCADA_REQUIRED = [
    "timestamp", "inverter_id", "dc_power_kw", "ac_power_kw",
    "dc_voltage_v", "dc_current_a", "module_temp_c",
    "inverter_efficiency", "status_code",
]

METEO_REQUIRED = [
    "station_id", "reading_time", "ghi_wm2", "ambient_temp", "temp_unit",
]

missing_scada = [c for c in SCADA_REQUIRED if c not in scada.columns]
missing_meteo = [c for c in METEO_REQUIRED if c not in meteo.columns]

if missing_scada:
    raise KeyError(f"Kolom SCADA tidak ditemukan: {missing_scada}")
if missing_meteo:
    raise KeyError(f"Kolom meteorological tidak ditemukan: {missing_meteo}")

scada["timestamp"] = pd.to_datetime(scada["timestamp"], errors="coerce")
meteo["reading_time"] = pd.to_datetime(meteo["reading_time"], errors="coerce")

invalid_scada_timestamp = int(scada["timestamp"].isna().sum())
invalid_meteo_timestamp = int(meteo["reading_time"].isna().sum())

scada = scada.dropna(subset=["timestamp"]).copy()
meteo = meteo.dropna(subset=["reading_time"]).copy()

print("Invalid SCADA timestamps removed:", invalid_scada_timestamp)
print("Invalid meteorological timestamps removed:", invalid_meteo_timestamp)
print("SCADA time range:", scada["timestamp"].min(), "->", scada["timestamp"].max())
print("Meteo time range:", meteo["reading_time"].min(), "->", meteo["reading_time"].max())


## 2. Validate numeric domains and current-unit consistency

The source column is named `dc_current_a`, but the numerical relationship in this source file is consistent with current being stored on a kA scale. For example, approximately `580 V × 0.1871 = 108.5 kW`. Therefore the canonical output converts the source numeric value to amperes by multiplying by 1000.

The notebook does not remove high-power observations based on this check. They remain available for the downstream anomaly model.

In [ ]:
numeric_scada = [
    "dc_power_kw", "ac_power_kw", "dc_voltage_v", "dc_current_a",
    "module_temp_c", "inverter_efficiency",
]
numeric_meteo = [
    "ghi_wm2", "dni_wm2", "dhi_wm2", "ambient_temp",
    "wind_speed_ms", "relative_humidity_pct", "barometric_pressure_hpa",
    "solar_zenith_angle_deg", "cloud_opacity_pct", "precipitation_mm",
]

for col in numeric_scada:
    scada[col] = pd.to_numeric(scada[col], errors="coerce")
for col in numeric_meteo:
    meteo[col] = pd.to_numeric(meteo[col], errors="coerce")

negative_checks = {
    "dc_power_kw": int((scada["dc_power_kw"] < 0).sum()),
    "ac_power_kw": int((scada["ac_power_kw"] < 0).sum()),
    "dc_voltage_v": int((scada["dc_voltage_v"] < 0).sum()),
    "dc_current_a": int((scada["dc_current_a"] < 0).sum()),
    "ghi_wm2": int((meteo["ghi_wm2"] < 0).sum()),
    "ambient_temp": int((meteo["ambient_temp"] < -80).sum()),
}

print("Negative/domain checks:")
print(pd.Series(negative_checks))

# Keep the original source value for auditability; canonical current is in amperes.
scada["source_dc_current_a"] = scada["dc_current_a"]
scada["dc_current_a"] = scada["dc_current_a"] * 1000.0

scada["dc_power_from_vi_kw"] = (
    scada["dc_voltage_v"] * scada["dc_current_a"] / 1000.0
)

power_denominator = scada["dc_power_from_vi_kw"].replace(0, np.nan)
scada["dc_power_consistency_ratio"] = (
    scada["dc_power_kw"] / power_denominator
)

operating = (
    scada["dc_power_kw"].notna()
    & scada["dc_power_from_vi_kw"].notna()
    & (scada["dc_power_from_vi_kw"] > 0)
)

ratio = scada.loc[operating, "dc_power_consistency_ratio"]
print("\nDC power/current physical consistency after unit normalization:")
print("Median P_DC / (V_DC*I_DC):", round(float(ratio.median()), 6))
print("P05:", round(float(ratio.quantile(0.05)), 6))
print("P95:", round(float(ratio.quantile(0.95)), 6))
print("Valid comparisons:", len(ratio))


## 3. Aggregate the three meteorological stations

Meteorological observations are available every 15 minutes from multiple stations. We first aggregate stations at each timestamp using the median, which avoids arbitrarily selecting one station. The resulting plant-level meteorological series is then resampled to 30-minute intervals to match SCADA.

In [ ]:
print("Weather stations:", sorted(meteo["station_id"].dropna().unique().tolist()))
print("Number of stations:", meteo["station_id"].nunique())

weather_numeric = [
    "ghi_wm2", "dni_wm2", "dhi_wm2", "ambient_temp",
    "wind_speed_ms", "relative_humidity_pct", "barometric_pressure_hpa",
    "solar_zenith_angle_deg", "cloud_opacity_pct", "precipitation_mm",
]

# Normalize obvious unit representation before aggregation.
meteo["temp_unit"] = meteo["temp_unit"].astype(str).str.upper().str.strip()
if not meteo["temp_unit"].dropna().isin(["C", "CELSIUS"]).all():
    raise ValueError("Meteorological temperature contains unsupported units. Convert to Celsius first.")

meteo_plant_15m = (
    meteo.groupby("reading_time", as_index=False)[weather_numeric]
    .median()
)

station_count = (
    meteo.groupby("reading_time")["station_id"]
    .nunique()
    .rename("weather_station_count")
)

meteo_plant_15m = meteo_plant_15m.merge(
    station_count,
    left_on="reading_time",
    right_index=True,
    how="left",
)

meteo_plant_15m = meteo_plant_15m.set_index("reading_time").sort_index()

# SCADA is 30-minute. Use time-based median for numeric weather variables.
meteo_plant_30m = meteo_plant_15m[weather_numeric].resample("30min").median()

# Keep station coverage as an audit variable. For each 30-min period, use the maximum
# observed number of stations contributing to the underlying 15-min records.
meteo_plant_30m["weather_station_count"] = (
    meteo_plant_15m["weather_station_count"].resample("30min").max()
)

meteo_plant_30m = meteo_plant_30m.reset_index()

print("Aggregated 15-min rows:", len(meteo_plant_15m))
print("Aggregated 30-min rows:", len(meteo_plant_30m))
display(meteo_plant_30m.head())


## 4. Align SCADA and plant-level meteorology

SCADA timestamps are already at 30-minute resolution. The join therefore uses the exact timestamp rather than a nearest-station lookup. Missing weather observations remain missing; they are not forward-filled or fabricated.

In [ ]:
scada = scada.sort_values(["timestamp", "inverter_id"]).reset_index(drop=True)
meteo_plant_30m = meteo_plant_30m.sort_values("reading_time").reset_index(drop=True)

dataset = scada.merge(
    meteo_plant_30m,
    left_on="timestamp",
    right_on="reading_time",
    how="left",
    validate="many_to_one",
)

dataset = dataset.drop(columns=["reading_time"])

dataset = dataset.rename(columns={
    "ghi_wm2": "irradiance_w_m2",
    "ambient_temp": "ambient_temp_c",
})

final_columns = [
    "timestamp",
    "inverter_id",
    "irradiance_w_m2",
    "ambient_temp_c",
    "module_temp_c",
    "dc_power_kw",
    "ac_power_kw",
    "dc_voltage_v",
    "dc_current_a",
    "inverter_efficiency",
    "status_code",
    "weather_station_count",
    "source_dc_current_a",
    "dc_power_from_vi_kw",
    "dc_power_consistency_ratio",
]

dataset = dataset[final_columns].sort_values(
    ["timestamp", "inverter_id"]
).reset_index(drop=True)

print("Final shape:", dataset.shape)
display(dataset.head(10))


## 5. Final validation

The checks below validate schema, uniqueness, missingness, physical consistency, and preservation of the original DC/AC SCADA variables. No anomaly/fault label is generated.

In [ ]:
print("=== DATASET VALIDATION ===")
print("Rows:", f"{len(dataset):,}")
print("Inverters:", dataset["inverter_id"].nunique())
print("Unique timestamps:", dataset["timestamp"].nunique())
print("Duplicate inverter-timestamp:", dataset.duplicated(["inverter_id", "timestamp"]).sum())

print("\nMissing values:")
display(dataset.isna().sum().to_frame("missing"))

print("\nCurrent unit conversion:")
print("Source column: dc_current_a")
print("Output column: dc_current_a")
print("Conversion: source value × 1000")

valid_ratio = dataset["dc_power_consistency_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
print("\nDC physical consistency ratio summary:")
display(valid_ratio.describe().to_frame("dc_power_consistency_ratio"))

print("\nAC/DC efficiency summary from source column:")
display(dataset["inverter_efficiency"].describe().to_frame("inverter_efficiency"))

print("\nWeather station coverage:")
display(dataset["weather_station_count"].value_counts(dropna=False).sort_index().to_frame("rows"))

print("\nPreserved SCADA variables:")
print([
    "dc_power_kw", "ac_power_kw", "dc_voltage_v", "dc_current_a",
    "module_temp_c", "inverter_efficiency", "status_code",
])


In [ ]:
dataset.to_csv(OUTPUT_FILE, index=False)

print(f"Validated dataset saved: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {dataset.columns.tolist()}")


## Methodological notes

1. The SCADA variables retain their original DC/AC semantics instead of collapsing them into generic `power_kw`, `voltage_v`, and `current_a` columns.
2. The numeric source `dc_current_a` is converted to canonical amperes because its observed scale is consistent with kA when checked against `dc_power_kw = dc_voltage_v × dc_current / 1000`.
3. `source_dc_current_a` is retained so the unit transformation remains auditable.
4. Three meteorological stations are aggregated by timestamp using the median before resampling to 30-minute plant-level data.
5. Weather alignment uses exact 30-minute timestamps and does not select one station arbitrarily.
6. Missing SCADA measurements are preserved. No interpolation or anomaly label is generated in this notebook.
7. `dc_power_from_vi_kw` and `dc_power_consistency_ratio` are diagnostic physical-consistency features; they are not fault labels.
8. Extreme values present in the source SCADA are retained so the downstream anomaly model can evaluate them rather than having the dataset builder remove them.
